# BERT Text Classifier

**Course:** Natural Language Processing · Unit 5 — Classification and Evaluation  
**Institution:** Universidad Tecnológica de Bolívar  
**Reference:** Jurafsky, D. & Martin, J. H. (2025). *Speech and Language Processing* (3rd ed.), Ch. 11 — Fine-Tuning and Masked Language Models  
**Python compatibility:** 3.10 · 3.11 · 3.12 · 3.13

---

In [ ]:
import transformers
print(transformers.__version__)

> **Output interpretation:** Confirms that the Transformers library is installed and shows its version. A version ≥ 4.30 is required for the `Trainer` API and dataset integration used in this notebook.

In [ ]:
# conda install -c huggingface -c conda-forge datasets  # run once

In [ ]:
from pathlib import Path

DIR_DATA = Path.cwd() / 'data'

In [ ]:
import os

os.makedirs("./logs", exist_ok=True)
os.makedirs("./results", exist_ok=True)
os.environ["WANDB_DISABLED"] = "true"

## Step 1: Importing Libraries

In [ ]:
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from transformers import (
    BertForSequenceClassification,
    BertTokenizer,
    Trainer,
    TrainingArguments,
)

## Step 2: Loading and Cleaning the Dataset

In [ ]:
df = pd.read_csv(DIR_DATA / "dataset_sentimientos_500.csv")

In [ ]:
df.columns = df.columns.str.strip()

In [ ]:
# 'Reseña' and 'Sentimiento' are the CSV column names (kept as-is)
df = df[['Reseña', 'Sentimiento']].dropna()
df['Sentimiento'] = df['Sentimiento'].map({'Positiva': 1, 'Negativa': 0})

In [ ]:
df

> **Output interpretation:** Displays the cleaned dataset with 500 reviews. The 'Sentimiento' column now shows binary labels (1 = Positive, 0 = Negative), confirming that the mapping was applied correctly and no null values remain.

## Step 3: Splitting into Training and Test Sets

In [ ]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['Reseña'].tolist(),
    df['Sentimiento'].tolist(),
    test_size=0.2,
    random_state=42,
)

## Step 4: Tokenization

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

train_encodings = tokenizer(
    train_texts, truncation=True, padding=True, max_length=128
)
test_encodings = tokenizer(
    test_texts, truncation=True, padding=True, max_length=128
)

## Step 5: Creating the Formal Datasets

In [ ]:
train_dataset = Dataset.from_dict({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask'],
    'labels': train_labels,
})

test_dataset = Dataset.from_dict({
    'input_ids': test_encodings['input_ids'],
    'attention_mask': test_encodings['attention_mask'],
    'labels': test_labels,
})

## Step 6: Defining Evaluation Metrics

In [ ]:
def compute_metrics(eval_pred):
    """Compute classification metrics from model predictions.

    Args:
        eval_pred: Tuple of (logits, labels) returned by the Trainer.

    Returns:
        dict with accuracy, F1, precision, and recall.
    """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary'
    )
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

## Step 7: Loading the Pre-trained Model

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=2
)

## Step 8: Configuring Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    logging_dir="./logs",
    logging_steps=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    evaluation_strategy="steps",  # evaluate at regular steps during training
    save_strategy="steps",        # checkpoint at regular steps
    report_to="none",             # disable W&B / external logging
)

## Step 9: Training the Model

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

## Step 10: Final Model Evaluation

In [ ]:
results = trainer.evaluate()
print("Results:", results)

> **Output interpretation:** Shows the final evaluation metrics on the held-out test set. Higher accuracy and F1 indicate that BERT has successfully learned to distinguish positive from negative sentiment; an F1 above 0.85 is typical for this binary task with 500 labelled examples.